# North Carolina — Chapter 58 (Insurance) → `data/north_carolina/ins_codes/*.md`

North Carolina’s **insurance** statutes are **Chapter 58** of the **General Statutes (N.C.G.S.)**. On **Justia**, the crawl root is **[`/codes/north-carolina/chapter-58/`](https://law.justia.com/codes/north-carolina/chapter-58/)**; sections live under **`…/chapter-58/article-…/section-58-…/`** (same structural pattern as **`new_mexico.ipynb`** / Chapter 59A).

**Cloudflare** often blocks plain **`httpx`**; this notebook uses **`curl_cffi`** with **`impersonate="chrome120"`**.

**Discovery:** BFS from the Chapter 58 index, following only paths under **`/codes/north-carolina/chapter-58/`** that are **not** **`/section-…`** pages; collect every section link (~**2,100** sections).

**Download:** text from **`div.primary-content`**, with Justia boilerplate stripped. Files are **`NC_gs_sec_<label>.md`** where **`<label>`** is the slug after **`section-`** with hyphens mapped to underscores (e.g. `58-1-1` → `NC_gs_sec_58_1_1.md`). The header includes a display citation like **`58-1-1`** (N.C.G.S. §).

**Config:** **`MAX_SECTIONS`**, **`MAX_DISCOVERY_PAGES`**, **`REUSE_DISCOVERED_URLS`**, **`_nc_chapter58_section_urls.txt`**.

Run with the **`ins_ipynb/`** directory as cwd. Then **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q curl_cffi beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
PATH_PREFIX = "/codes/north-carolina/chapter-58"
TITLE_INDEX = f"{BASE}{PATH_PREFIX}/"

OUT_DIR = Path("data") / "north_carolina" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

MAX_SECTIONS = 0
MAX_DISCOVERY_PAGES = 0

SKIP_EXISTING = True

DISCOVERED_LIST = OUT_DIR / "_nc_chapter58_section_urls.txt"
REUSE_DISCOVERED_URLS = True


In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def discover_section_urls() -> list[str]:
    """BFS chapter-58 index + article pages; collect section URLs."""
    from collections import deque

    start = TITLE_INDEX
    seen: set[str] = set()
    in_q: set[str] = {path_key(start).lower()}
    q: deque[str] = deque([start])
    sections: set[str] = set()
    fetches = 0
    while q:
        if MAX_DISCOVERY_PAGES and fetches >= MAX_DISCOVERY_PAGES:
            break
        url = q.popleft()
        pk = path_key(url).lower()
        in_q.discard(pk)
        if pk in seen:
            continue
        if "/section-" in pk.lower():
            continue
        seen.add(pk)
        html = curl_get(url)
        fetches += 1
        soup = BeautifulSoup(html, "html.parser")
        for a in soup.find_all("a", href=True):
            absu = urljoin(url, a["href"])
            p = path_key(absu).lower()
            if not p.startswith(PATH_PREFIX):
                continue
            if "/section-" in p.lower():
                sections.add(BASE + p + "/")
            else:
                if p in seen or p in in_q:
                    continue
                in_q.add(p)
                q.append(BASE + p + "/")
    return sorted(sections)


def section_label_from_url(url: str) -> str:
    path = path_key(url)
    low = path.lower()
    if "/section-" not in low:
        raise ValueError(f"not a section URL: {url!r}")
    return path.rsplit("/section-", 1)[1]


def label_sort_key(label: str) -> tuple:
    out: list[tuple[int, int | str]] = []
    for part in label.split("-"):
        if part.isdigit():
            out.append((0, int(part)))
        else:
            out.append((1, part.lower()))
    return tuple(out)


def label_to_display_citation(label: str) -> str:
    """58-1-1 -> 58-1-1 ; 58a-3-10 -> 58A-3-10"""
    parts = label.split("-")
    out: list[str] = []
    for p in parts:
        if len(p) >= 2 and p[:-1].isdigit() and p[-1].isalpha():
            out.append(p[:-1] + p[-1].upper())
        else:
            out.append(p)
    return "-".join(out)


def label_to_filename(label: str) -> str:
    safe = label.replace("-", "_")
    return f"NC_gs_sec_{safe}.md"


def extract_primary_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
    )
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    for line in lines:
        s = line.strip()
        if not s:
            if not skip_until_substantive:
                out.append("")
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s.startswith("20") and ("N.C. Gen" in s or "North Carolina Gen" in s):
            continue
        if s in {"Next", "Previous", "Universal Citation:"}:
            continue
        if s.startswith("N.C. Gen. Stat") or s.startswith("NC Gen Stat"):
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_chapter_58() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and DISCOVERED_LIST.exists() and DISCOVERED_LIST.stat().st_size > 50:
        raw = [ln.strip() for ln in DISCOVERED_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        all_urls = sorted(raw, key=lambda u: label_sort_key(section_label_from_url(u)))
        print(f"Loaded {len(all_urls)} section URLs from {DISCOVERED_LIST.name} (skipped discovery)")
    else:
        found = discover_section_urls()
        print(f"Discovered {len(found)} section URLs under Chapter 58")
        all_urls = sorted(found, key=lambda u: label_sort_key(section_label_from_url(u)))
        DISCOVERED_LIST.write_text("\n".join(all_urls) + "\n", encoding="utf-8")

    todo = all_urls if not MAX_SECTIONS else all_urls[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited downloads to first {len(todo)} sections (MAX_SECTIONS)")

    wrote = skipped = failed = 0
    for i, sec_url in enumerate(todo, 1):
        label = section_label_from_url(sec_url)
        disp = label_to_display_citation(label)
        dest = OUT_DIR / label_to_filename(label)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                html = curl_get(sec_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                title = head_t or f"North Carolina General Statutes § {disp}"
                md = (
                    f"# {title}\n\n"
                    f"**North Carolina General Statutes — Chapter 58 (Insurance)**\n\n"
                    f"**Source (Justia mirror):** {sec_url}\n\n"
                    f"**Verify on official site:** [NC General Assembly — Statutes](https://www.ncleg.gov/Laws/GeneralStatutes)\n\n"
                    f"**Section (URL slug):** {label}\n\n"
                    f"**Citation (display):** N.C.G.S. § {disp}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {label}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_chapter_58()


Discovered 2112 section URLs under Chapter 58
… 200/2112 (wrote=200 skipped=0 failed=0)
… 400/2112 (wrote=400 skipped=0 failed=0)
… 600/2112 (wrote=600 skipped=0 failed=0)
… 800/2112 (wrote=800 skipped=0 failed=0)
… 1000/2112 (wrote=1000 skipped=0 failed=0)
… 1200/2112 (wrote=1200 skipped=0 failed=0)
… 1400/2112 (wrote=1400 skipped=0 failed=0)
… 1600/2112 (wrote=1600 skipped=0 failed=0)
… 1800/2112 (wrote=1800 skipped=0 failed=0)
… 2000/2112 (wrote=2000 skipped=0 failed=0)
Done. wrote=2112 skipped=0 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/north_carolina/ins_codes


{'wrote': 2112, 'skipped': 0, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
